# 01 · v3 Baseline — Physics CV + LGBM 잔차 + W1 가중 (EXP #10, LB 0.6678)

대회 baseline이자 이후 모든 lever의 출발점. **등속(CV) 외삽**을 물리 base로 깔고, 그 잔차(`y − base`)를 LGBM이 학습한다.

**가정**: 잔차 모델에 boundary(1cm 경계) 근처 sample을 더 무겁게 학습시키는 **W1 Gaussian 가중**(σ=3.5mm, max_w=2.5, EXP #6·#8에서 튜닝)이 HR을 직접 끌어올린다. 단 in-sample 가중은 leakage라 **OOF residual로 가중을 시드**해야 하고, 8000/2000 holdout 대신 **전체 10000 × 5-fold 재학습**이 데이터를 최대로 쓴다.

**실험**: Phase 1(no-weight 5-fold)로 OOF 잔차 norm을 구해 W1 가중 입력으로 쓰고 → Phase 2(W1 가중 5-fold)로 재학습 + test 예측 평균. v1(8000만)/v2(W1·5-fold 누락)와 비교해 "전체 10000 + W1 + 5-fold" 조합의 효과를 격리한다.

**결과**: **LB 0.6678** (v1 0.664 대비 +0.0038). 데이터 volume(전체 재학습)이 W1 sample-weight 미세튜닝보다 큰 lever임을 확인. 이후 모든 lever는 이 v3 위에서 평가한다.

## 셀 0 — imports + 상수

In [ ]:
import os, time, json
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold

DT, DT_PRED = 0.04, 0.08
SEED, N_FOLDS = 42, 5
MINORITY_THRESHOLD = 5.0
HIT_RADIUS = 0.01

# D4 채택 파라미터 (results/d4_best.json)
BEST_SIGMA = 0.0035
BEST_MAX_W = 2.5

os.makedirs('results', exist_ok=True)
print(f'D4 채택: σ={BEST_SIGMA*1000}mm, max_w={BEST_MAX_W}')
print(f'SEED={SEED}, N_FOLDS={N_FOLDS}, HIT_RADIUS={HIT_RADIUS}')

## 셀 1 — helper 함수 (D4와 비트 단위 동일 + make_splits_full)

In [ ]:
FEAT_NAMES = [
    'vx_last', 'vy_last', 'vz_last', 'ax_last', 'ay_last', 'az_last',
    'speed_last', 'acc_norm_last',
    'ax_w3', 'ay_w3', 'az_w3', 'acc_norm_w3',
    'vx_std', 'vy_std', 'vz_std', 'ax_std', 'ay_std', 'az_std', 'path_eff',
    'distance_r', 'radial_v',
    'turn_mean', 'cos_turn_last', 'va_dot',
    'jerk_x_last', 'jerk_y_last', 'jerk_z_last',
    'a_tang_last', 'a_cent_last',
    'speed_diff_half', 'turn_mean_half_diff',
]
assert len(FEAT_NAMES) == 31


def _norm(x):
    return np.linalg.norm(x, axis=-1)


def physics_baseline(traj):
    p0, p_m40 = traj[:, -1, :], traj[:, -2, :]
    v_last = (p0 - p_m40) / DT
    return p0 + v_last * DT_PRED


def build_features(traj):
    p = traj
    v = (p[:, 1:, :] - p[:, :-1, :]) / DT
    a = (v[:, 1:, :] - v[:, :-1, :]) / DT
    v_last, a_last = v[:, -1, :], a[:, -1, :]
    speed_last, acc_norm_last = _norm(v_last), _norm(a_last)
    w = np.array([1, 2, 3]) / 6
    a3 = a[:, -3:, :]
    a_w3 = (a3 * w[None, :, None]).sum(axis=1)
    acc_norm_w3 = _norm(a_w3)
    v_std, a_std = v.std(axis=1), a.std(axis=1)
    seg = _norm(p[:, 1:, :] - p[:, :-1, :])
    path_eff = _norm(p[:, -1, :] - p[:, 0, :]) / (seg.sum(1) + 1e-9)
    p0 = p[:, -1, :]
    distance_r = _norm(p0)
    radial_v = (p0 * v_last).sum(1) / (distance_r + 1e-9)
    v_n = v / (_norm(v)[..., None] + 1e-9)
    cos_consec = (v_n[:, 1:, :] * v_n[:, :-1, :]).sum(-1).clip(-1, 1)
    turn = np.arccos(cos_consec)
    turn_mean, cos_turn_last = turn.mean(1), cos_consec[:, -1]
    va_dot = (v_last * a_last).sum(1) / (speed_last * acc_norm_last + 1e-9)
    jerk_last = (a[:, -1, :] - a[:, -2, :]) / DT
    v_a_dot = (v_last * a_last).sum(1)
    v_cross_a = np.cross(v_last, a_last)
    a_tang_last = v_a_dot / (speed_last + 1e-9)
    a_cent_last = _norm(v_cross_a) / (speed_last + 1e-9)
    speed_seq = _norm(v)
    speed_diff_half = speed_seq[:, 5:].mean(1) - speed_seq[:, :5].mean(1)
    turn_mean_half_diff = turn[:, 4:].mean(1) - turn[:, :4].mean(1)
    feats = np.column_stack([
        v_last, a_last, speed_last, acc_norm_last,
        a_w3, acc_norm_w3,
        v_std, a_std, path_eff,
        distance_r, radial_v,
        turn_mean, cos_turn_last, va_dot,
        jerk_last,
        a_tang_last, a_cent_last,
        speed_diff_half, turn_mean_half_diff,
    ]).astype(np.float32)
    return feats


def hit_rate(pred, label, r=HIT_RADIUS):
    return (np.linalg.norm(pred - label, axis=1) < r).mean()


def w_gaussian(e, sigma, max_w, r=HIT_RADIUS):
    return np.clip(1.0 + (max_w - 1.0) * np.exp(-(e - r)**2 / (2 * sigma**2)), 0.5, max_w)


def make_splits_full(minority_int, seed=SEED):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    return [(tr, va) for tr, va in skf.split(np.arange(len(minority_int)), minority_int)]


print('helper 정의 완료')

## 셀 2 — Drive mount + zip 해제 (train + test 동시 확보)

In [ ]:
CACHE_TRAJ_TR = 'traj_train.npy'
CACHE_Y_TR    = 'y_train.npy'
CACHE_TRAJ_TS = 'traj_test.npy'

DATA_DIR = None
if not (os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR) and os.path.exists(CACHE_TRAJ_TS)):
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP_PATH = '/content/drive/MyDrive/open.zip'
    !unzip -q -o "{ZIP_PATH}" -d /content/
    DATA_DIR = '/content/open' if os.path.exists('/content/open/sample_submission.csv') else '/content'
    print(f'DATA_DIR = {DATA_DIR}')
else:
    print('모든 캐시 존재 — Drive mount 생략')

## 셀 3 — train 로딩 (캐시 우선)

In [ ]:
import pandas as pd

if os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR):
    traj_train = np.load(CACHE_TRAJ_TR)
    y_train    = np.load(CACHE_Y_TR)
    print(f'train cache: traj {traj_train.shape}, y {y_train.shape}')
else:
    labels = pd.read_csv(f'{DATA_DIR}/train_labels.csv')
    train_ids = labels['id'].tolist()
    y_train = labels[['x','y','z']].values.astype(np.float32)
    traj_train = np.empty((len(train_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(train_ids):
        df = pd.read_csv(f'{DATA_DIR}/train/{tid}.csv')
        traj_train[i] = df[['x','y','z']].values
    np.save(CACHE_TRAJ_TR, traj_train)
    np.save(CACHE_Y_TR, y_train)
    print(f'train fresh: traj {traj_train.shape}, y {y_train.shape}')

assert traj_train.shape == (10000, 11, 3)
assert y_train.shape == (10000, 3)
assert np.isfinite(traj_train).all() and np.isfinite(y_train).all()

## 셀 4 — test 로딩 (sample_submission 순서 보존)

In [ ]:
if DATA_DIR is None:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    if not os.path.exists('/content/open/sample_submission.csv') and not os.path.exists('/content/sample_submission.csv'):
        ZIP_PATH = '/content/drive/MyDrive/open.zip'
        !unzip -q -o "{ZIP_PATH}" -d /content/
    DATA_DIR = '/content/open' if os.path.exists('/content/open/sample_submission.csv') else '/content'

sample_sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
test_ids = sample_sub['id'].tolist()
assert len(test_ids) == 10000

if os.path.exists(CACHE_TRAJ_TS):
    traj_test = np.load(CACHE_TRAJ_TS)
    print(f'test cache: traj {traj_test.shape}')
else:
    traj_test = np.empty((len(test_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(test_ids):
        df = pd.read_csv(f'{DATA_DIR}/test/{tid}.csv')
        traj_test[i] = df[['x','y','z']].values
    np.save(CACHE_TRAJ_TS, traj_test)
    print(f'test fresh: traj {traj_test.shape}')

assert traj_test.shape == (10000, 11, 3)
assert np.isfinite(traj_test).all()

## 셀 5 — physics baseline + residual + features (train + test 동시)

In [ ]:
base_train = physics_baseline(traj_train)
base_test  = physics_baseline(traj_test)
resid_train = (y_train - base_train).astype(np.float32)

X_train = build_features(traj_train)
X_test  = build_features(traj_test)

acc_mag_tr = X_train[:, FEAT_NAMES.index('acc_norm_last')]
minority_mask_tr = acc_mag_tr >= MINORITY_THRESHOLD

assert X_train.shape == (10000, 31)
assert X_test.shape  == (10000, 31)
assert np.isfinite(X_train).all() and np.isfinite(X_test).all()

print(f'physics HR (train): {hit_rate(base_train, y_train):.4f}  (목표 ≈ 0.5788)')
print(f'X_train {X_train.shape}, X_test {X_test.shape}')
print(f'train minority ratio: {minority_mask_tr.mean():.3f}')

## 셀 6 — 5-fold split (전체 10000, stratify=minority)

진짜 Option B: tr/va holdout 없이 전체 10000을 5-fold로만 분할. D4 결과 va_idx HR 0.6665는 8000+2000 protocol이라 v3에서 직접 재현 X — sanity는 OOF HR로 확인.

In [ ]:
folds = make_splits_full(minority_mask_tr.astype(int), seed=SEED)
for k, (tr_in, val_in) in enumerate(folds):
    print(f'fold {k}: tr {len(tr_in)}, val {len(val_in)}, val_minority {minority_mask_tr[val_in].mean():.3f}')

## 셀 7 — Phase 1: D1-equivalent OOF residual on full 10000

W1 weight 입력으로 쓸 OOF residual norm. no-weight 5-fold × 3 axis = 15 모델. ~1.5분.

In [ ]:
lgbm_params = dict(
    objective='regression_l1', metric='mae',
    learning_rate=0.05, num_leaves=31, min_data_in_leaf=5,
    max_bin=511, n_estimators=500, random_state=SEED, verbose=-1,
)

oof_resid_pred = np.full((10000, 3), np.nan, dtype=np.float32)

t0 = time.time()
for k, (tr_in, val_in) in enumerate(folds):
    for ax in range(3):
        m = lgb.LGBMRegressor(**lgbm_params)
        m.fit(X_train[tr_in], resid_train[tr_in, ax],
              eval_set=[(X_train[val_in], resid_train[val_in, ax])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        oof_resid_pred[val_in, ax] = m.predict(X_train[val_in]).astype(np.float32)
print(f'Phase 1 (D1 OOF on full 10000) done in {time.time()-t0:.1f}s')

assert np.isfinite(oof_resid_pred).all()

oof_e_full = np.linalg.norm(resid_train - oof_resid_pred, axis=1).astype(np.float32)
pred_oof = base_train + oof_resid_pred
oof_hr_full      = float(hit_rate(pred_oof, y_train))
majority_hr_oof  = float(hit_rate(pred_oof[~minority_mask_tr], y_train[~minority_mask_tr]))
minority_hr_oof  = float(hit_rate(pred_oof[minority_mask_tr], y_train[minority_mask_tr]))

print(f'\n[Phase 1 sanity — no-weight OOF on full 10000]')
print(f'  OOF HR overall : {oof_hr_full:.4f}  (D1 tr_idx 8000 OOF 0.6430 참고)')
print(f'  OOF HR majority: {majority_hr_oof:.4f}')
print(f'  OOF HR minority: {minority_hr_oof:.4f}')
print(f'  ||resid_OOF|| mean   : {oof_e_full.mean()*1000:.2f}mm  (D1 11.90mm 참고)')
print(f'  ||resid_OOF|| median : {np.median(oof_e_full)*1000:.2f}mm')

np.save('results/v3_oof_resid_pred.npy', oof_resid_pred)
np.save('results/v3_oof_e.npy', oof_e_full)

## 셀 8 — Phase 2: D4 5-fold W1 weight on full 10000 + test 예측

각 fold에서:
1. fold tr_in의 Phase 1 OOF residual norm으로 W1 weight 계산
2. weighted LGBM 학습 × 3 axis
3. test 10000 예측 누적 (5-fold 평균)

15 모델, ~1.5분.

In [ ]:
pred_test_resid = np.zeros((10000, 3), dtype=np.float32)
oof_resid_d4    = np.full((10000, 3), np.nan, dtype=np.float32)

t0 = time.time()
for k, (tr_in, val_in) in enumerate(folds):
    w_tr = w_gaussian(oof_e_full[tr_in], BEST_SIGMA, BEST_MAX_W)
    print(f'fold {k}: w_tr range [{w_tr.min():.3f}, {w_tr.max():.3f}], mean {w_tr.mean():.3f}')
    for ax in range(3):
        m = lgb.LGBMRegressor(**lgbm_params)
        m.fit(X_train[tr_in], resid_train[tr_in, ax],
              sample_weight=w_tr,
              eval_set=[(X_train[val_in], resid_train[val_in, ax])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        oof_resid_d4[val_in, ax]  = m.predict(X_train[val_in]).astype(np.float32)
        pred_test_resid[:, ax]   += m.predict(X_test).astype(np.float32) / N_FOLDS
print(f'\nPhase 2 (D4 W1 on full 10000 + test prediction) done in {time.time()-t0:.1f}s')

assert np.isfinite(oof_resid_d4).all()
assert np.isfinite(pred_test_resid).all()

pred_oof_d4     = base_train + oof_resid_d4
oof_hr_d4       = float(hit_rate(pred_oof_d4, y_train))
majority_hr_d4  = float(hit_rate(pred_oof_d4[~minority_mask_tr], y_train[~minority_mask_tr]))
minority_hr_d4  = float(hit_rate(pred_oof_d4[minority_mask_tr], y_train[minority_mask_tr]))

print(f'\n[Phase 2 sanity — D4 W1 OOF on full 10000]')
print(f'  OOF HR overall : {oof_hr_d4:.4f}  (Phase 1 no-weight {oof_hr_full:.4f} 대비 {oof_hr_d4-oof_hr_full:+.4f})')
print(f'  OOF HR majority: {majority_hr_d4:.4f}')
print(f'  OOF HR minority: {minority_hr_d4:.4f}')

np.save('results/v3_oof_resid_d4.npy', oof_resid_d4)
np.save('results/v3_pred_test_resid.npy', pred_test_resid)

## 셀 9 — submission_d4_v3.csv 생성

In [ ]:
pred_test_abs = base_test + pred_test_resid

assert pred_test_abs.shape == (10000, 3)
assert np.isfinite(pred_test_abs).all()

submission_df = pd.DataFrame({
    'id': test_ids,
    'x': pred_test_abs[:, 0],
    'y': pred_test_abs[:, 1],
    'z': pred_test_abs[:, 2],
})

submission_path = 'submission_d4_v3.csv'
submission_df.to_csv(submission_path, index=False)

print(f'{submission_path} 저장 완료 ({len(submission_df)} rows)')
print(submission_df.head(3))
print(f'\npred_test_abs 통계 (per axis):')
for i, name in enumerate(['x','y','z']):
    a = pred_test_abs[:, i]
    print(f'  {name}: mean {a.mean():.3f}, std {a.std():.3f}, range [{a.min():.3f}, {a.max():.3f}]')

## 셀 10 — v3 메타데이터 저장 + Drive 복사 + 로컬 다운로드

In [ ]:
with open('results/v3_meta.json', 'w') as f:
    json.dump({
        'best_sigma': BEST_SIGMA,
        'best_max_w': BEST_MAX_W,
        'n_train': 10000,
        'n_folds': N_FOLDS,
        'phase1_oof_hr': oof_hr_full,
        'phase1_oof_e_mean_mm': float(oof_e_full.mean()*1000),
        'phase2_oof_hr_overall':  oof_hr_d4,
        'phase2_oof_hr_majority': majority_hr_d4,
        'phase2_oof_hr_minority': minority_hr_d4,
        'd4_va_idx_overall_ref': 0.6665,
        'lb_v1_ref': 0.664,
        'lb_v2_ref': 0.664,
    }, f, indent=2)
print('results/v3_meta.json 저장')

from google.colab import drive, files
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_BASE_PATH = '/content/drive/MyDrive'
results_dir_drive = os.path.join(DRIVE_BASE_PATH, 'results')
os.makedirs(results_dir_drive, exist_ok=True)

!cp -r results/* {results_dir_drive}/
!cp {submission_path} {DRIVE_BASE_PATH}/
print(f'results/* -> {results_dir_drive} 복사 완료')
print(f'{submission_path} -> {DRIVE_BASE_PATH}/ 복사 완료')
!ls -la {results_dir_drive}/v3_*

files.download(submission_path)